# UN VOTES DATABASE

Author and maintainer: Vadim Rudakov, rudakow.wadim@gmail.com

The “UN votes” Database is a collection of normalized data of the UN voted resolutions’ details from 1946 to present, collected from the [UN Digital Library](https://digitallibrary.un.org/search?c=Voting+Data&cc=Voting+Data&ln=en). The database is intended as a source for machine learning experiments with international relations data, such as building predictive models for future votes or clustering countries. That is why the main goal of the database is the countries’ vote results. Not all the resolutions presented have this information, but we decided to include the details on all available resolutions for those researchers who would want to conduct other type of analysis where the vote results do not play significant role.

The database is updated monthly to incorporate newly voted resolutions. See [RELEASE_NOTES.md](RELEASE_NOTES.md) for the current version statistics.

# Quick Start

Download the latest `un_votes.sql.gz` from [Releases](https://github.com/soviar-systems/un_votes/releases).

## Using Podman (recommended)

No local PostgreSQL installation required — everything runs inside the container. The database is automatically created and populated on first start:

``` bash
# Start PostgreSQL and restore the dump (one command)
podman run -d --name un-votes-postgres \
  -e POSTGRES_DB=un_votes \
  -e POSTGRES_USER=user1 \
  -e POSTGRES_PASSWORD=12345 \
  -v ./un_votes.sql.gz:/docker-entrypoint-initdb.d/un_votes.sql.gz:Z \
  docker.io/library/postgres:17

# Connect to the database
podman exec -it un-votes-postgres psql -U user1 -d un_votes
```

In [1]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT * FROM un.resolution
 LIMIT 5;"

   id   | title_id |       symbol        | meeting_record_id | vote_date  
--------+----------+---------------------+-------------------+------------
 671311 |        1 | A/RES/63(I)[PARA.8] |                 1 | 1946-12-13
 671195 |        2 | A/RES/181(II)[A]    |                 2 | 1947-11-29
 671194 |        3 | A/RES/180(II)       |                 3 | 1947-11-21
 671193 |        4 | A/RES/179(II)[C]    |                 3 | 1947-11-21
 671192 |        4 | A/RES/179(II)[B]    |                 3 | 1947-11-21
(5 rows)



The first start takes about a minute while PostgreSQL processes the dump. To stop and remove the container when done: `podman rm -f un-votes-postgres`

> **How it works:** the official PostgreSQL container image automatically executes any `.sql`, `.sql.gz`, or `.sh` files found in the `/docker-entrypoint-initdb.d/` directory on first startup. By mounting our dump there, the database is created and populated without any manual steps. This only happens once — subsequent container starts skip the initialization if data already exists.

## Using an existing PostgreSQL cluster (advanced)

If you already have a running PostgreSQL server and know how to administer it:

``` bash
# Create a user and database (run as a PostgreSQL superuser, e.g. postgres)
psql -U postgres -c "CREATE USER user1 WITH PASSWORD '12345';"
psql -U postgres -c "CREATE DATABASE un_votes WITH OWNER user1;"

# Restore the dump
gunzip -c un_votes.sql.gz | psql -U user1 -d un_votes

# Connect
psql -U user1 -d un_votes
```

## Verify the installation

Run the built-in summary function:

``` sql
SELECT * FROM un.get_database_statistics();
```

We will show the real examples here using the direct connection to the database in the container:

In [2]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT * FROM un.get_database_statistics();"

      metric      |   value   
------------------+-----------
 years            | 1946–2026
 resolutions      | 10111
 resolutions (GA) | 7371
 resolutions (SC) | 2740
 recorded         | 8440
 non-recorded     | 1671
 countries        | 241
 votes            | 988852
(8 rows)



# Part I. How the database was collected and built

This document covers the technical details of how the database was collected and how you can work with it to get the most out of it.

The entire project of building the database can be divided into 3 big parts.

1.  Analysis of the data provided by the UN Digital Library and the database architecture elaboration. We had to decide:

- what kind of data was accessible to us and what kind of data should have really been stored for our purposes,
- how to organize this data to make it easy 1) to update it with the new data, and 2) to work with for analysts,
- what naming and data types we should choose for the database schemata.

“[Architecture](#Architecture)” gives answers on how we solved this problems.

1.  Next step involved gathering, processing, and sending the data from the UN Digital Library website to the PostgreSQL database.

For this task we wrote the crawler using the popular Python’s “scrapy” module. For sending the data to the database we used another Python’s module “psycopg 3”. The processing pipeline was written in a way to treat each scrapy’s item (i.e. all the details of one resolution) as a transaction, within which all the operations of inserting data and getting the foreign keys for already inserted data either completed successfully or aborted completely. This allowed the maintainer to be sure that all the data was consistent and keep track of bad transactions to rewrite the code during the development and test stage.

1.  And the last part was to maintain the system of logs.

This step was essential to have a right to release the database to the community - we could not share the data we did not trust ourselves. Logs system was to control the correctness of two processes: 1) sending the data to the database (client side) and 2) storing the data in the database (server side), and make us be able to trace any kind of corrupted data during the database development step to make changes in the source code. We do not open our source code to prevent misuse that could overload the UN Digital Library servers, but we share all the logs so everyone can see the entire way of the data from the website to the database. All the scraping and sending data to the database job was automated, no hand work was implemented at all to exclude any kind of human made mistake.

# Part II. Working with the database

## Architecture

To get the full out of the database, one should understand its architecture. If you have worked with the relational databases before and have the familiarity with the `JOIN` operation, you will quickly grasp the idea of how to work with the database. The maintainers intended to implement the classic “many-to-many” model.

The architecture idea of the database comes from two sources:
- the resolution’s available details,
- the strive for the database normalization, i.e. the principle “one string in one place” realization.

This is the example of the typical webpage with the resolution details:

![Figure 1. UN Resolution page on the UN Digital Library website with the details](./images/Screenshot_20240311_185702.png)

*Figure 1. UN Resolution page on the UN Digital Library website with the details*

On the Fig. 2 you can see the attributes that have been processed to the database:

![Figure 2. Attributes that came into the database on the UN Digital Library website with the details](./images/Screenshot_20240311_185702_1.png)

*Figure 2. Attributes that came into the database on the UN Digital Library website with the details*

As you may have noticed, no information on “Draft”, “Note” and “Vote summary” has been deemed valuable for our purposes. These data contain no additional useful information but would have overloaded the database and slowed down its performance unnecessarily. Regarding the “Vote summary” field, this information can be easily derived from the `vote` table where we store all the votes country by country, so this information should not take its own storage space. Probably, the only reason we should somehow incorporate this information is that “NON-RECORDED” resolutions, i.e. resolutions without country-by-country votes, cannot be identified as accepted or rejected within our database. If there is a real need for such information, we will rewrite the crawler and rebuild the database, but for now this information is not included.

All the highlighted fields keep their names in the database, so you can easily switch between the web-page and the query result when needed.

> [!NOTE]
> **Why the crawler filters with `fct__9=Vote`**
>
> The UN Digital Library’s [Voting Data collection](https://digitallibrary.un.org/search?c=Voting+Data&cc=Voting+Data&ln=en) contains ~23,500 records in total, but only ~10,000 of them are typed as “Vote”. The remaining ~13,500 are resolutions adopted without vote (by consensus). The crawler uses the `fct__9=Vote` search facet to collect only the records where a voting procedure took place.
>
> The UN uses three adoption methods:
>
> 1.  **Recorded vote** — country-by-country roll-call, our primary data.
> 2.  **Non-recorded vote** — show of hands; we know the totals but no per-country breakdown. These are in the database with 0 rows in the `vote` table.
> 3.  **Adopted without vote** — consensus, no voting procedure at all. These are the ~13,500 excluded records.
>
> The `fct__9=Vote` facet captures categories 1 and 2, which is exactly the right boundary for a vote analysis database. Category 3 records have no vote data whatsoever — including them would add thousands of rows to the `resolution` table with no analytical value for vote pattern research.

This is the database final architecture:

```mermaid
erDiagram
    TITLE ||--o{ RESOLUTION : "has"
    MEETING_RECORD ||--o{ RESOLUTION : "associated with"
    RESOLUTION ||--o{ RESOLUTION_AGENDA : "linked to"
    AGENDA ||--o{ RESOLUTION_AGENDA : "linked to"
    RESOLUTION ||--o{ RESOLUTION_COMMITTEE_REPORT : "linked to"
    COMMITTEE_REPORT ||--o{ RESOLUTION_COMMITTEE_REPORT : "linked to"
    RESOLUTION ||--o{ VOTE : "has"
    COUNTRY ||--o{ VOTE : "casts"
    VOTE_CHOICE ||--o{ VOTE : "defines"
    
    RESOLUTION {
        int id PK
        int title_id FK
        string symbol
        int meeting_record_id FK
        date vote_date
    }
    TITLE {
        int id PK
        string name
    }
    MEETING_RECORD {
        int id PK
        string symbol
    }
    AGENDA {
        int id PK
        string name
    }
    RESOLUTION_AGENDA {
        int resolution_id FK
        int agenda_id FK
    }
    COMMITTEE_REPORT {
        int id PK
        string symbol
    }
    RESOLUTION_COMMITTEE_REPORT {
        int resolution_id FK
        int committee_report_id FK
    }
    COUNTRY {
        int id PK
        string name
    }
    VOTE_CHOICE {
        int id PK
        string choice
    }
    VOTE {
        int resolution_id FK
        int country_id FK
        int vote_choice_id FK
    }
    README {
        int id PK
        string lang
        string title
        string description
        string contact_info
    }
```

*Figure 3. UN Votes Database architecture*

## Tables

Now let’s take a look at the tables and their interaction with each other.

There are 10 tables in the database that contain the UN resolutions data, and also there is an additional bilingual table with the general info about the database (`readme`, filtered by `lang` column: `'en'` or `'ru'`). This is the list of the the UN data containg tables:

1.  `agenda`
2.  `committee_report`
3.  `country`
4.  `meeting_record`
5.  `resolution`
6.  `resolution_agenda`
7.  `resolution_committee_report`
8.  `title`
9.  `vote`
10. `vote_choice`

You can see the entire list of the tables using command `\dt`, and also you can use `\d` to see the schema of each table, for example:

In [3]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\dt un.*"

                  List of relations
 Schema |            Name             | Type  | Owner 
--------+-----------------------------+-------+-------
 un     | agenda                      | table | user1
 un     | committee_report            | table | user1
 un     | country                     | table | user1
 un     | meeting_record              | table | user1
 un     | readme                      | table | user1
 un     | resolution                  | table | user1
 un     | resolution_agenda           | table | user1
 un     | resolution_committee_report | table | user1
 un     | title                       | table | user1
 un     | vote                        | table | user1
 un     | vote_choice                 | table | user1
(11 rows)



## “resolution” table

The `resolution` table is the core table - its `resolution.id` attribute binds all the tables together. But if you compare the resolution attributes on the page in the Fig. 1 and the names of tables in the tables’ list you will notice that some of these attributes have their own tables and some don’t. The `resolution` table has only 5 columns:
- `id` - record number that you can use to quickly find the web page by substituting the “RECORD” word in the url `https://digitallibrary.un.org/record/RECORD?ln=en` with this id (for example, the resolution from the Figure 1 has `id = 278340` and its web address is, then, “https://digitallibrary.un.org/record/278340?ln=en”)
- `title_id`,
- `symbol` is the UN documentation standard, see https://research.un.org/en/docs/symbols for details,
- `meeting_record_id`,
- `vote_date`.

In [4]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\d un.resolution"

                           Table "un.resolution"
      Column       |         Type          | Collation | Nullable | Default 
-------------------+-----------------------+-----------+----------+---------
 id                | integer               |           | not null | 
 title_id          | integer               |           |          | 
 symbol            | character varying(64) |           |          | 
 meeting_record_id | integer               |           |          | 
 vote_date         | date                  |           |          | 
Indexes:
    "resolution_pkey" PRIMARY KEY, btree (id)
    "month_b" btree (EXTRACT(month FROM vote_date))
    "resolution_symbol_key" UNIQUE CONSTRAINT, btree (symbol)
    "year_b" btree (EXTRACT(year FROM vote_date))
Foreign-key constraints:
    "resolution_meeting_record_id_fkey" FOREIGN KEY (meeting_record_id) REFERENCES un.meeting_record(id) ON DELETE CASCADE
    "resolution_title_id_fkey" FOREIGN KEY (title_id) REFERENCES un.title(id) ON DELE

### How resolution’s attributes are stored

The `resolution` table has only a few attributes because some resolutions’ attributes may have more than one distinct value. There are only three such attributes: `agenda`, `committee_report`, and `vote`, all of them have gotten their own tables for data integrity purposes (see details how to use them in later sections).

Consider example for the resolution with id `518324`. It has ten (!) values for agenda:

![Figure 5. Eight values for [agenda](https://digitallibrary.un.org/record/518324?ln=en)](./images/Screenshot_20260427_204309.jpeg)

*Figure 5. Eight values for [agenda](https://digitallibrary.un.org/record/518324?ln=en)*

Here’s the record in our database, no agenda column:

In [5]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT * FROM un.resolution 
WHERE id = 518324;"

   id   | title_id |      symbol      | meeting_record_id | vote_date  
--------+----------+------------------+-------------------+------------
 518324 |     7140 | S/RES/1534(2004) |              7140 | 2004-03-26
(1 row)



We send all 8 agenda values to the `agenda` table where each new agenda value gets its unique id:

In [6]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT * FROM un.agenda 
WHERE name ~* 'S/59.*65.*croatia situation.*';"

  id  |            name             
------+-----------------------------
 9327 | S/59 [65] CROATIA SITUATION
(1 row)



Then we update an intermediary table called `resolution_agenda` that have two columns - `resolution_id` (the foreign key to `resolution.id`) and `agenda_id` (the foreign key to `agenda.id`).

Thus, we get all the agendas for the given resolution in the intermediate table `resolution_agenda`:

In [7]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT * FROM un.resolution_agenda 
WHERE resolution_id = 518324;"

 resolution_id | agenda_id 
---------------+-----------
        518324 |      7518
        518324 |      7519
        518324 |      9211
        518324 |      9212
        518324 |      9235
        518324 |      9324
        518324 |      9327
        518324 |      9328
        518324 |      9329
        518324 |      9330
(10 rows)



In [8]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT a.name AS agenda_name
FROM un.resolution_agenda ra
JOIN un.agenda a ON ra.agenda_id = a.id
JOIN un.resolution r ON ra.resolution_id = r.id
JOIN un.title t ON r.title_id = t.id
WHERE ra.resolution_id = 518324;"

                                                                                                                                                                               agenda_name                                                                                                                                                                               
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 S/ X International Tribunal for the Prosecution of Persons Responsible for Serious Violations of International Humanitarian Law Committed in the Territory of the Former Yugoslavia since 1991.
 S/ X International Criminal Tribunal for the Prosecution of Persons Responsible fo

In [9]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT * FROM un.agenda WHERE id = 9327
LIMIT 5;"

  id  |            name             
------+-----------------------------
 9327 | S/59 [65] CROATIA SITUATION
(1 row)



Here you can see the top 10 resolutions with the largest number of `agenda` values:

In [10]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- Top 10 resolutions with the most agenda items
SET search_path TO un;
SELECT
    resolution_id,
    count(resolution_id) AS cnt   -- how many agenda items this resolution has
FROM resolution_agenda
GROUP BY resolution_id
ORDER BY cnt DESC
LIMIT 10;"

SET
 resolution_id | cnt 
---------------+-----
        518324 |  10
        816559 |   6
        721811 |   6
       1324646 |   6
        285170 |   6
        418338 |   6
       4053365 |   6
        853289 |   6
        283223 |   6
        285563 |   6
(10 rows)



and the same rating for `committee_report`:

In [11]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- Top 10 resolutions with the most committee reports
SET search_path TO un;
SELECT
    resolution_id,
    count(resolution_id) AS cnt   -- how many committee reports this resolution has
FROM resolution_committee_report
GROUP BY resolution_id
ORDER BY cnt DESC
LIMIT 10;"

SET
 resolution_id | cnt 
---------------+-----
        670969 |   5
        667226 |   4
        670663 |   4
        663787 |   3
        671166 |   3
        664380 |   3
        670297 |   3
        279828 |   3
        663789 |   3
        671254 |   3
(10 rows)



### JOIN all the attributes

Now we can `JOIN` these tables to get the the original information (you can also change the format view to resemble the website format adding `\gx` to the end of the command in terminal):

In [12]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SET search_path TO un;
-- All GA resolutions with full details (no votes)
SELECT
    r.id AS record,               -- resolution ID, also matches UN Digital Library record ID
    t.name AS title,              -- resolution title
    a.name AS agenda,             -- agenda item text
    r.symbol AS resolution,       -- UN document symbol (A/RES/...)
    mr.symbol AS meeting_record,  -- meeting record symbol
    cr.symbol AS committee_report,-- committee report symbol
    r.vote_date                   -- date the vote took place
FROM resolution r
JOIN title t ON r.title_id = t.id
JOIN resolution_agenda ra ON r.id = ra.resolution_id
JOIN agenda a ON ra.agenda_id = a.id
JOIN meeting_record mr ON r.meeting_record_id = mr.id
JOIN resolution_committee_report rc ON r.id = rc.resolution_id
JOIN committee_report cr ON rc.committee_report_id = cr.id
WHERE r.symbol ~ '^A'             -- only General Assembly resolutions
ORDER BY r.vote_date DESC
LIMIT 5;"

SET
 record  |                                                                                          title                                                                                          |                                                                                                agenda                                                                                                 |  resolution  | meeting_record | committee_report | vote_date  
---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+----------------+------------------+------------
 4108513 | Revised estimates relating to the programme budget for 2026 under

This command returned all the attributes for the resolutions voted in the General Assembly, except for the vote, for each resolution in the descending order.

And this command will return the same attributes but also the vote of the Egypt:

In [13]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SET search_path TO un;
-- All GA resolutions with Egypt's vote in each
SELECT
    r.id AS record,
    t.name AS title,
    a.name AS agenda,
    r.symbol AS resolution,
    mr.symbol AS meeting_record,
    cr.symbol AS committee_report,
    r.vote_date,
    vc.choice AS egypt_vote       -- resolved vote_choice_id to 'yes'/'no'/'abstentions'/'non-voting'
FROM resolution r
JOIN title t ON r.title_id = t.id
JOIN resolution_agenda ra ON r.id = ra.resolution_id
JOIN agenda a ON ra.agenda_id = a.id
JOIN meeting_record mr ON r.meeting_record_id = mr.id
JOIN resolution_committee_report rc ON r.id = rc.resolution_id
JOIN committee_report cr ON rc.committee_report_id = cr.id
JOIN vote v ON r.id = v.resolution_id
JOIN vote_choice vc ON v.vote_choice_id = vc.id
WHERE r.symbol ~ '^A'
  AND v.country_id = (            -- subquery: find Egypt's country_id by name (case-insensitive)
      SELECT id FROM country WHERE name ~* '.*egypt.*'
  )
ORDER BY r.vote_date DESC
LIMIT 5;"

SET
 record  |                                                                                          title                                                                                          |                                                                                                agenda                                                                                                 |  resolution  | meeting_record | committee_report | vote_date  | egypt_vote 
---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+----------------+------------------+------------+------------
 4108513 | Revised estimates relating to the progr

If you’re struggling in understanding this syntax, please, consider learning SQL basic commands and relational databases design models (for example: “[PostgreSQL for Everybody](https://www.coursera.org/specializations/postgresql-for-everybody)”).

## “agenda” table

The `agenda` table contains two columns:

In [14]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\d un.agenda"

                                Table "un.agenda"
 Column |  Type   | Collation | Nullable |                Default                
--------+---------+-----------+----------+---------------------------------------
 id     | integer |           | not null | nextval('un.agenda_id_seq'::regclass)
 name   | text    |           |          | 
Indexes:
    "agenda_pkey" PRIMARY KEY, btree (id)
    "agenda_name_key" UNIQUE CONSTRAINT, btree (name)
Referenced by:
    TABLE "un.resolution_agenda" CONSTRAINT "resolution_agenda_agenda_id_fkey" FOREIGN KEY (agenda_id) REFERENCES un.agenda(id) ON DELETE CASCADE



Figure 1 shows no special `subject` field but if you carefully look at the `agenda` string for the resolutions adopted after 1983, you will notice that the last part of the string is often capitalized:

![Figure 8. Agenda strings newer than 1983 examples](./images/Screenshot_20240323_185659.png)

*Figure 8. Agenda strings newer than 1983 examples*

The capitalized portion is a **subject** that describes the `agenda` in a more general manner. Previously, we extracted this into a separate `subject` table, but the extraction was unreliable (the UN does not use a consistent format — early resolutions have no subject at all, and Security Council agenda strings follow entirely different patterns). Since the subject is derivable from the agenda string itself, maintaining it as a separate table was a normalization violation with no practical benefit.

> **Working with Subjects:** If you need subject-level filtering, you can perform pattern-based filtering on `agenda.name` using SQL. Subjects are often embedded within the `agenda.name` field (typically after a double dash `--`). Since the format is inconsistent, we recommend using `LIKE` or `ILIKE` patterns for specific categories (e.g., `WHERE name ILIKE '%--%DISARMAMENT%'`). For official UN subject taxonomies, please refer to the [UN Digital Library search page](https://digitallibrary.un.org/search?cc=Voting%20Data&ln=en&p=&f=&rm=&sf=&so=d&rg=50&c=Voting%20Data&c=&of=hb&fti=0&fti=0).

## “committee_report” table

This table contains two columns:

In [15]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\d un.committee_report"

                                       Table "un.committee_report"
 Column |         Type          | Collation | Nullable |                     Default                     
--------+-----------------------+-----------+----------+-------------------------------------------------
 id     | integer               |           | not null | nextval('un.committee_report_id_seq'::regclass)
 symbol | character varying(64) |           |          | 
Indexes:
    "committee_report_pkey" PRIMARY KEY, btree (id)
    "committee_report_symbol_key" UNIQUE CONSTRAINT, btree (symbol)
Referenced by:
    TABLE "un.resolution_committee_report" CONSTRAINT "resolution_committee_report_committee_report_id_fkey" FOREIGN KEY (committee_report_id) REFERENCES un.committee_report(id) ON DELETE CASCADE



One resolution may have:
- zero,
- one,
- many

`committee_report` values.

That is why the additional - `resolution_committee_report` - table was created, designed for binding `resolution.id`s and their corresponding `committee_report.id` values:

In [16]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT * FROM un.resolution_committee_report LIMIT 10;"

 resolution_id | committee_report_id 
---------------+---------------------
        671311 |                   1
        671195 |                   2
        671194 |                   3
        671193 |                   4
        671193 |                   5
        671192 |                   4
        671192 |                   5
        671191 |                   4
        671191 |                   5
        671189 |                  10
(10 rows)



## “country” table

In [17]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\d un.country"

                                Table "un.country"
 Column |  Type   | Collation | Nullable |                Default                 
--------+---------+-----------+----------+----------------------------------------
 id     | integer |           | not null | nextval('un.country_id_seq'::regclass)
 name   | text    |           |          | 
Indexes:
    "country_pkey" PRIMARY KEY, btree (id)
    "country_name_key" UNIQUE CONSTRAINT, btree (name)
Referenced by:
    TABLE "un.vote" CONSTRAINT "vote_country_id_fkey" FOREIGN KEY (country_id) REFERENCES un.country(id) ON DELETE CASCADE



The crawler processes data year by year starting from 1946 to present, so countries appear in the `country` table roughly in the order they first voted in the UN bodies (General Assembly and Security Council). This can be used as a rough indicator of when a country joined the UN, but should not be taken as the ultimate truth — the order depends on which resolution within a year the country first appears in.

## “meeting_record” table

In [18]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\d un.meeting_record"

                                       Table "un.meeting_record"
 Column |         Type          | Collation | Nullable |                    Default                    
--------+-----------------------+-----------+----------+-----------------------------------------------
 id     | integer               |           | not null | nextval('un.meeting_record_id_seq'::regclass)
 symbol | character varying(64) |           |          | 
Indexes:
    "meeting_record_pkey" PRIMARY KEY, btree (id)
    "meeting_record_symbol_key" UNIQUE CONSTRAINT, btree (symbol)
Referenced by:
    TABLE "un.resolution" CONSTRAINT "resolution_meeting_record_id_fkey" FOREIGN KEY (meeting_record_id) REFERENCES un.meeting_record(id) ON DELETE CASCADE



This table contains two columns:
- `id`, and
- `symbol` (UN Documentation standard).

There can only be one `meeting_record` value for the resolution, but many resolutions may share the same `meeting_record`.

## “title” table

In [19]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\d un.title"

                                Table "un.title"
 Column |  Type   | Collation | Nullable |               Default                
--------+---------+-----------+----------+--------------------------------------
 id     | integer |           | not null | nextval('un.title_id_seq'::regclass)
 name   | text    |           |          | 
Indexes:
    "title_pkey" PRIMARY KEY, btree (id)
    "title_name_key" UNIQUE CONSTRAINT, btree (name)
Referenced by:
    TABLE "un.resolution" CONSTRAINT "resolution_title_id_fkey" FOREIGN KEY (title_id) REFERENCES un.title(id) ON DELETE CASCADE



There can only be one `title` value for the resolution, but many resolutions may share the same `title`.

## “vote” table

In [20]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\d un.vote"

                      Table "un.vote"
     Column     |   Type   | Collation | Nullable | Default 
----------------+----------+-----------+----------+---------
 resolution_id  | integer  |           |          | 
 country_id     | integer  |           |          | 
 vote_choice_id | smallint |           |          | 
Indexes:
    "vote_resolution_id_country_id_vote_choice_id_key" UNIQUE CONSTRAINT, btree (resolution_id, country_id, vote_choice_id)
Foreign-key constraints:
    "vote_country_id_fkey" FOREIGN KEY (country_id) REFERENCES un.country(id) ON DELETE CASCADE
    "vote_resolution_id_fkey" FOREIGN KEY (resolution_id) REFERENCES un.resolution(id) ON DELETE CASCADE
    "vote_vote_choice_id_fkey" FOREIGN KEY (vote_choice_id) REFERENCES un.vote_choice(id) ON DELETE CASCADE



This table has been the main purpose of the database - each country’s vote results that you can use for interesting analyses.

This table contains only foreign keys:
- `resolution_id`,
- `country_id`,
- `vote_choice_id`

that you can use to get the vote of the country you are interested in for the given resolution.

## “vote_choice” table

In [21]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\d un.vote_choice"

                        Table "un.vote_choice"
  Column   |          Type           | Collation | Nullable | Default 
-----------+-------------------------+-----------+----------+---------
 id        | smallint                |           | not null | 
 choice    | character varying(16)   |           |          | 
 choice_ru | character varying(1000) |           |          | 
Indexes:
    "vote_choice_pkey" PRIMARY KEY, btree (id)
    "vote_choice_choice_key" UNIQUE CONSTRAINT, btree (choice)
    "vote_choice_choice_ru_key" UNIQUE CONSTRAINT, btree (choice_ru)
Referenced by:
    TABLE "un.vote" CONSTRAINT "vote_vote_choice_id_fkey" FOREIGN KEY (vote_choice_id) REFERENCES un.vote_choice(id) ON DELETE CASCADE



This additional table contains only 4 rows - one row for each possible vote choice: ‘yes’, ‘no’, ‘abstentions’, and ‘non-voting’. If you look at the Fig. 1 example, you will see that the `vote` field uses empty string for non-voting countries and ‘Y’, ‘N’ and ‘A’ for other variants. We processed these values, changed them to the numbers on our choice, and created this table with the explanation of the vote result:

In [22]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"SELECT * FROM un.vote_choice;"

 id |   choice    |    choice_ru     
----+-------------+------------------
  0 | no          | против
  1 | yes         | за
  2 | abstentions | воздержавшиеся
  3 | non-voting  | без права голоса
(4 rows)



If you have any questions or suggestions, feel free to contact the maintainer.

# Part III. SQL queries examples

Use `\pset format wrapped` command to get the lines in console wrapped for the better view experience.

## Database summary information

See [Quick Start](#quick-start) for usage of `un.get_database_statistics()` and a sample query.

## “JOIN” all details, except for “vote”

In [23]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- All resolutions with full details (both GA and SC, no votes)
SET search_path TO un;
SELECT
    r.id AS record,
    t.name AS title,
    a.name AS agenda,
    r.symbol AS resolution,
    mr.symbol AS meeting_record,
    cr.symbol AS committee_report,
    r.vote_date
FROM resolution r
JOIN title t ON r.title_id = t.id
JOIN resolution_agenda ra ON r.id = ra.resolution_id
JOIN agenda a ON ra.agenda_id = a.id
JOIN meeting_record mr ON r.meeting_record_id = mr.id
JOIN resolution_committee_report rc ON r.id = rc.resolution_id
JOIN committee_report cr ON rc.committee_report_id = cr.id
LIMIT 5;"

SET
 record |                                                                                                                                                       title                                                                                                                                                       |                                                                                                                                            agenda                                                                                                                                            | resolution  | meeting_record | committee_report | vote_date  
--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------

## “JOIN” all details only for GA, except for “vote”

In [24]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- Same as above, filtered to General Assembly resolutions only
SET search_path TO un;
SELECT
    r.id AS record,
    t.name AS title,
    a.name AS agenda,
    r.symbol AS resolution,
    mr.symbol AS meeting_record,
    cr.symbol AS committee_report,
    r.vote_date
FROM resolution r
JOIN title t ON r.title_id = t.id
JOIN resolution_agenda ra ON r.id = ra.resolution_id
JOIN agenda a ON ra.agenda_id = a.id
JOIN meeting_record mr ON r.meeting_record_id = mr.id
JOIN resolution_committee_report rc ON r.id = rc.resolution_id
JOIN committee_report cr ON rc.committee_report_id = cr.id
WHERE r.symbol ~ '^A'            -- GA resolutions start with 'A'
LIMIT 5;"

SET
 record |                                                                                                                                                       title                                                                                                                                                       |                                                                                                                                            agenda                                                                                                                                            | resolution  | meeting_record | committee_report | vote_date  
--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------

## “JOIN” “vote” results of one “country” for each resolution

For example Egypt, in descending order by date

In [25]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- All GA resolutions with Egypt's vote
SET search_path TO un;
SELECT
    r.id AS record,
    t.name AS title,
    a.name AS agenda,
    r.symbol AS resolution,
    mr.symbol AS meeting_record,
    cr.symbol AS committee_report,
    r.vote_date,
    vc.choice AS egypt_vote       -- 'yes', 'no', 'abstentions', or 'non-voting'
FROM resolution r
JOIN title t ON r.title_id = t.id
JOIN resolution_agenda ra ON r.id = ra.resolution_id
JOIN agenda a ON ra.agenda_id = a.id
JOIN meeting_record mr ON r.meeting_record_id = mr.id
JOIN resolution_committee_report rc ON r.id = rc.resolution_id
JOIN committee_report cr ON rc.committee_report_id = cr.id
JOIN vote v ON r.id = v.resolution_id
JOIN vote_choice vc ON v.vote_choice_id = vc.id
WHERE r.symbol ~ '^A'
  AND v.country_id = (
      SELECT id FROM country WHERE name ~* '.*egypt.*'
  )
ORDER BY r.vote_date DESC
LIMIT 5;"

SET
 record  |                                                                                          title                                                                                          |                                                    agenda                                                    |  resolution  | meeting_record | committee_report | vote_date  | egypt_vote 
---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------+--------------+----------------+------------------+------------+------------
 4108513 | Revised estimates relating to the programme budget for 2026 under section 3, Political affairs, and section 36, Staff assessment : resolution / adopted by the General Assembly         | A/80/251 136 Proposed programme

## Get the records from a specific year

In [26]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- All resolutions voted in 2024
SELECT * FROM un.resolution
WHERE EXTRACT(YEAR FROM vote_date) = 2024
LIMIT 5;"

   id    | title_id |      symbol      | meeting_record_id | vote_date  
---------+----------+------------------+-------------------+------------
 4070472 |     9718 | A/RES/79/256     |              9718 | 2024-12-24
 4070464 |      350 | A/RES/79/249     |              9718 | 2024-12-24
 4070387 |     9956 | S/RES/2767(2024) |              9956 | 2024-12-27
 4070358 |     9957 | S/RES/2760(2024) |              9957 | 2024-11-14
 4070039 |     9958 | A/RES/79/240     |              9718 | 2024-12-24
(5 rows)



## Count the number of values in a resolution’s field

For example, let’s count the number of `committee_report` values for each resolution in descending order:

In [27]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- How many committee reports each resolution has
SELECT
    resolution_id,
    COUNT(*) AS cnt               -- number of committee reports
FROM un.resolution_committee_report
GROUP BY resolution_id
ORDER BY cnt DESC
LIMIT 5;"

 resolution_id | cnt 
---------------+-----
        670969 |   5
        667226 |   4
        670663 |   4
        663789 |   3
        279828 |   3
(5 rows)



and filter only those resolutions that contain more than 2 `committee_report` values:

In [28]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- Resolutions with more than 2 committee reports
SELECT
    resolution_id,
    COUNT(*) AS cnt
FROM un.resolution_committee_report
GROUP BY resolution_id
HAVING COUNT(*) > 2               -- filter: keep only those with 3+ reports
ORDER BY cnt DESC
LIMIT 5;"

 resolution_id | cnt 
---------------+-----
        670969 |   5
        670663 |   4
        667226 |   4
        670297 |   3
        663789 |   3
(5 rows)



## Show “agenda” from a particular year

Let’s see `agenda` in the General Assembly resolutions since 1983:

In [29]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- Agenda items for GA resolutions from 1983 onward
SET search_path TO un;
SELECT
    r.id AS resolution,
    a.name AS agenda,
    r.vote_date
FROM agenda a
JOIN resolution_agenda ra ON a.id = ra.agenda_id
JOIN resolution r ON r.id = ra.resolution_id
WHERE EXTRACT(YEAR FROM r.vote_date) >= 1983
  AND r.symbol ~* '^a'            -- General Assembly only
ORDER BY r.vote_date
LIMIT 5;"

SET
 resolution |                                                                                                                                            agenda                                                                                                                                            | vote_date  
------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------
     625796 | A/37/251 37 Question of Cyprus.                                                                                                                                                                                                                                                              | 1983-05-13
     278340 | A/38/251 23 Situation in Kampuchea. KAMPUCHE

and for the Security Council since 1985:

In [30]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"-- Agenda items for SC resolutions from 1985 onward
SET search_path TO un;
SELECT
    r.id AS resolution,
    a.name AS agenda,
    r.vote_date
FROM agenda a
JOIN resolution_agenda ra ON a.id = ra.agenda_id
JOIN resolution r ON r.id = ra.resolution_id
WHERE EXTRACT(YEAR FROM r.vote_date) >= 1985
  AND r.symbol ~* '^s'            -- Security Council only
ORDER BY r.vote_date
LIMIT 5;"

SET
 resolution |                                                                             agenda                                                                              | vote_date  
------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+------------
     279769 | S/40 [10] APARTHEID                                                                                                                                             | 1985-03-12
     279770 | S/40 [17] MIDDLE EAST SITUATION                                                                                                                                 | 1985-04-17
     279770 | S/40 [19] UN INTERIM FORCE IN LEBANON                                                                                                                           | 1985-04-17
     279771 | S/ X Letter dated 6 May 1985 from the Permane

## Save the result in csv

For example, save the `agenda` topics by year (extracting `subject` from agenda strings):

``` sql
-- Export subjects by year to CSV
-- Note: only works for agenda strings that use the standardized format
\copy (
    SELECT DISTINCT ON (a.name)
        substring(a.name FROM '^\S+\s+\d+[a-z]?\s+([^:]+):') AS subject,  -- extract text between item number and colon
        EXTRACT(YEAR FROM r.vote_date) AS year
    FROM agenda a
    JOIN resolution_agenda ra ON a.id = ra.agenda_id
    JOIN resolution r ON r.id = ra.resolution_id
    WHERE a.name ~ '^\S+\s+\d+[a-z]?\s+[^:]+:'    -- only strings with extractable subject
    ORDER BY a.name, year
) TO '~/UN_Analysis/subjects.csv' WITH CSV HEADER;
```

> **Note:** The standardized agenda format (`A/660 32 Subject : subtitle`) became common only from 1983 onward. Earlier resolutions do not have extractable subjects in this format — the regex filter `WHERE a.name ~ '...'` in the query above excludes them automatically.

# Misc

## Indexes

Two B-Tree expression indexes are created automatically by the pipeline after each crawl:

``` sql
CREATE INDEX IF NOT EXISTS year_b ON resolution(EXTRACT(YEAR FROM vote_date));
CREATE INDEX IF NOT EXISTS month_b ON resolution(EXTRACT(MONTH FROM vote_date));
```

These speed up queries that filter by year or month (e.g. `WHERE EXTRACT(YEAR FROM vote_date) = 2024`). A plain B-tree on `vote_date` would not help such queries — PostgreSQL cannot use a column index to satisfy an expression predicate.

In [31]:
podman exec un-votes-postgres psql -U user1 -d un_votes -c \
"\di un.*"  | grep -E 'year_b|month_b'

 un     | month_b                                                         | index | user1 | resolution
 un     | year_b                                                          | index | user1 | resolution


<!-- #endregion -->